In [ ]:
!uv pip install -q pyngrok
!uv pip install -U huggingface_hub

In [ ]:
!wget -O vllm-0.26.1rc1.dev1212+k2.cu128-cp312-cp312-linux_x86_64.whl "https://github.com/iqzaardiansyah/vllm-cuda-builds/releases/download/k2-git-d9fd5f114/vllm-0.26.1rc1.dev1212.k2.cu128-cp312-cp312-linux_x86_64.whl"
!pip -q install ./vllm-0.26.1rc1.dev1212+k2.cu128-cp312-cp312-linux_x86_64.whl

In [ ]:
import os
import time
import subprocess
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

try:
    user_secrets = UserSecretsClient()
    NGROK_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")
except Exception as e:
    print("ERROR: Could not find 'NGROK_AUTH_TOKEN' in Kaggle Secrets.", flush=True)
    raise e

print("Opening ngrok tunnel...", flush=True)
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000, host_header="localhost").public_url

model_path = "iqzaardiansyah/K2-Horizon-7B-AWQ-4bit-64g"

print(f"\nStarting vLLM server with {model_path}...", flush=True)
BASE_TOKENIZER = "IFM/K2-Horizon-7B"

command = f"""
vllm serve {model_path} \
  --tokenizer {BASE_TOKENIZER} \
  --tensor-parallel-size 2 \
  --dtype float16 \
  --max-model-len auto \
  --gpu-memory-utilization 0.95 \
  --max-num-seqs 16 \
  --enable-prefix-caching \
  --reasoning-parser k2_horizon \
  --enable-auto-tool-choice \
  --tool-call-parser k2_horizon \
  --trust-remote-code \
  --served-model-name K2-Horizon-7B-GGUF \
  --port 8000
"""

server_process = subprocess.Popen(command, shell=True)

print("\n" + "="*65, flush=True)
print("vLLM SERVER IS STARTING")
print("Wait for weights to download and shard.")
print("Base URL for your coding agent:")
print(f"{public_url}/v1", flush=True)
print("="*65 + "\n", flush=True)

try:
    print("Server is active. Keep this browser tab open!", flush=True)
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nShutting down...", flush=True)
    server_process.terminate()
    ngrok.kill()